In [1]:
import json
import struct
import numpy as np
import matplotlib.pyplot as plt
import photonforge as pf
import siepic_forge as siepic
import luxtelligence_lnoi400_forge as lxt
import tidy3d as td

td.config.logging.level = "ERROR"

# Set up technologies
siepic_tech = siepic.ebeam()
basic_tech = pf.basic_technology()
lxt_tech = lxt.lnoi400()
pf.config.default_technology = siepic_tech

# Initialize live viewer for real-time visualization
from photonforge.live_viewer import LiveViewer
viewer = LiveViewer()

# Define simulation parameters
wavelengths = np.linspace(1.53, 1.57, 101)
freqs = pf.C_0 / wavelengths

21:23:29 SE Asia Standard Time WARNING: Using canonical configuration directory 
                               at 'C:\Users\James\.config\tidy3d'. Found legacy 
                               directory at 'C:\Users\James\.tidy3d', which will
                               be ignored. Tidy3D configuration now uses        
                               'C:\Users\James\.config\tidy3d\config.toml'.     

21:23:39 SE Asia Standard Time WARNING: The material-library variant            
                               'Palik_Lossless' is deprecated and maps to       
                               'Palik_LowLoss' because it contains a tiny fitted
                               loss despite its name. Use 'Palik_NoLoss' where  
                               available for a zero-loss Palik model.           

LiveViewer started at http://localhost:55294


In [3]:
dual_mode_spec = siepic_tech.ports["TE_1550_500"].copy()
dual_mode_spec.num_modes = 2  # Use both modes

siepic_tech.add_port("TE-TM_1550_500", dual_mode_spec)
siepic_tech.ports["TE-TM_1550_500"]

PortSpec(description="Strip TE 1550 nm, w=500 nm", width=1.5, limits=(-0.6, 0.82), num_modes=2, added_solver_modes=0, polarization="", target_neff=3.5, default_radius=0, path_profiles=[(0.5, 0, (1, 0))])

In [4]:

@pf.parametric_component
def create_y_branch(port_spec="TE-TM_1550_500", input_length=10, output_length=20, offset_up=5, offset_down=5):
    port_width = 0.5

    component = pf.Component("y_branch")

    sbend_up = pf.parametric.s_bend(port_spec=port_spec, length=output_length, offset=offset_up)
    sbend_down = pf.parametric.s_bend(port_spec=port_spec, length=output_length, offset=-offset_down)

    taper = pf.stencil.linear_taper(input_length, [port_width, port_width*2])
    component.add("Si", taper)

    sb1_ref = component.add_reference(sbend_up)
    sb2_ref = component.add_reference(sbend_down)
    sb1_ref.x_min = taper.x_max
    sb1_ref.y_min = taper.y_max - port_width
    sb2_ref.x_min = taper.x_max
    sb2_ref.y_max = taper.y_min + port_width

    port_symmetries = [
        ("P0", "P2", "P1"),
    ]

    field_monitor = td.FieldMonitor(
        center=(0, 0, 0.11), size=(td.inf, td.inf, 0), freqs=[freqs.mean()], name="field"
    )

    component.add_port(component.detect_ports([port_spec], on_boundary="x"))
    component.add_model(pf.Tidy3DModel(port_symmetries=port_symmetries, monitors=[field_monitor]), "Tidy3DModel")

    return component

# weird
y_branch = create_y_branch()
viewer(y_branch)


c:\Users\James\AppData\Local\Programs\Python\Python313\Lib\site-packages\photonforge\parametric_utils.py:198: RuntimeWarning: Component function '__main__.create_y_branch' previously registered will be overwritten.
  return _decorator(decorated_function)
